# 03 — Vision Redaction via AWS Bedrock
Send each page image to Claude 3.7 Sonnet on Bedrock.

PII/PHI is **replaced with realistic fictitious dummy values** (not blanked).
The same original value gets the **same replacement** across all pages of a document.

Outputs per page stored in `redacted_text/` as `{stem}_page_{n}.json`:
```json
{ "sanitized_text": "...", "mapping": [{"original_masked": ..., "replacement": ..., "type": ...}] }
```

In [ ]:
%pip install pymupdf boto3 Pillow reportlab --prefer-binary -q

In [ ]:
import boto3, json, base64, os, time
from pathlib import Path
from collections import defaultdict

from utils import get_logger, extract_json
logger = get_logger("03_redact_via_bedrock")

BASE_DIR      = Path(os.getcwd()).parent
TEMP_FOLDER   = BASE_DIR / "temp_images"
TEXT_FOLDER   = BASE_DIR / "redacted_text"

AWS_REGION    = "us-east-2"
BEDROCK_MODEL = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
MAX_TOKENS    = 4096
RETRY_LIMIT   = 3
RETRY_DELAY   = 5

TEXT_FOLDER.mkdir(exist_ok=True)
client = boto3.client("bedrock-runtime", region_name=AWS_REGION)
logger.info("Bedrock client ready — model: %s", BEDROCK_MODEL)

In [ ]:
BASE_PROMPT = """CRITICAL RULE: Every replacement value you generate MUST be a completely different invented \
name/value. If you read "MARGARET HOLMES" in the document, the replacement CANNOT be "MARGARET HOLMES" \
or any variation of it — it must be an entirely different fictitious value like "DIANA CHEN". If the \
original SSN is "456-78-9012", the replacement cannot be "456-78-9012" — it must be a different \
made-up number like "731-29-8854". A replacement that matches the original is a critical failure.

You are a data-sanitization assistant. You will be given an image of a document page. \
Your job is to produce a rewritten version of that page's text in which every PII/PHI value has \
been physically replaced with a made-up fictitious substitute. The output text must NOT contain \
any of the original sensitive values — they must be gone, replaced by different invented values.

EXAMPLE OF WHAT YOU MUST DO:
  Original text:  "Claimant: John Smith   SSN: 123-45-6789   DOB: 03/14/1972"
  sanitized_text: "Claimant: Alex Rivera  SSN: 987-65-4321   DOB: 07/22/1985"
  mapping: [
    {"original_masked": "J*** S***",     "replacement": "Alex Rivera",  "type": "Name"},
    {"original_masked": "***-**-6789",   "replacement": "987-65-4321",  "type": "SSN"},
    {"original_masked": "**/**/1972",    "replacement": "07/22/1985",   "type": "DOB"}
  ]
Notice: the sanitized_text contains the REPLACEMENT values, not the originals. \
If the original name was "John Smith", the word "John Smith" must NOT appear anywhere \
in sanitized_text — it must be replaced by "Alex Rivera" (or whatever fictitious name you chose).

Redact and replace ONLY these PII/PHI categories:
- Full names in ANY format or order: "First Last", "Last, First", "Last, First Middle", titles \
(Mr., Dr., etc.), suffixes (Jr., Sr., III), and initials. This includes names preceded by role \
labels such as "Claimant:", "Patient:", "Provider:", "Insured:", "Applicant:", "Member:", \
"Beneficiary:", etc.
- Email addresses
- Phone and fax numbers
- Social Security Numbers (SSN) or national identifiers
- Dates of Birth (DOB only — not date of service, date of injury, or any other dates)
- Medical record numbers or identifiers (MRN, patient ID, chart number, etc.)
- Medical diagnoses or conditions tied to individuals (disease names, ICD codes, clinical descriptions)
- Credit card details (card numbers, expiration dates, CVVs, cardholder names alongside card data)

Do NOT redact or replace anything outside those categories. Leave unchanged:
- Physical mailing addresses
- Insurance, policy, group, or claim numbers
- Dates other than DOB (date of service, date of injury, admission/discharge dates, etc.)
- Driver's license or state ID numbers
- Bank account or routing numbers
- Employer names or job titles
- Facility names, hospital names, or clinic names

Requirements:
1. Transcribe ALL text from the page — including headings, labels, table contents, and footers.
2. As you transcribe, substitute every PII/PHI value with your invented replacement. \
The replacement MUST be a completely different made-up value that shares NO words with the original. \
"MARGARET HOLMES" → "DIANA CHEN" is correct. "MARGARET HOLMES" → "MARGARET HOLMES" is a critical failure. \
"MARGARET HOLMES" → "MARGARET CHEN" is also wrong — no words may overlap.
3. If the same original value appears multiple times, always use the same replacement for it.
4. Replacements must follow realistic formats (names look like names, SSNs look like SSNs, etc.) \
but must not correspond to real individuals.
5. Pay special attention to names inside table cells or form fields, names in "Last, First" \
inverted format, and names following role labels (e.g. "Claimant:", "Patient:"). \
These are PII regardless of formatting or context.
6. In the mapping, "original_masked" is a partially obscured hint of the original \
(e.g. "J*** S***" for "John Smith") — never write the full original value there. \
"replacement" is the invented value you wrote into sanitized_text. \
They must always be different.

<<PRIOR_MAPPING>>
SELF-CHECK — do this before returning your response:
1. For every row in your mapping, verify that "replacement" is completely different from the \
original value you saw in the document. If they match or closely resemble each other, discard \
that replacement and invent a new one.
2. Scan your sanitized_text and confirm that NONE of the original PII/PHI values from the \
document appear anywhere in it. If any original value leaked through, replace it with the \
corresponding fictitious value from your mapping.

Return ONLY valid JSON with exactly this structure (no markdown fences, no extra keys):
{
  "sanitized_text": "<complete transcription of the page with all PII/PHI replaced — originals must not appear>",
  "mapping": [
    {
      "original_masked": "<partially obscured original, e.g. J*** S***>",
      "replacement": "<the invented value written into sanitized_text>",
      "type": "<Name | SSN | DOB | Email | Phone | MRN | Diagnosis | CreditCard>"
    }
  ]
}

Now transcribe and sanitize the following document page:"""

BEDROCK_IMAGE_LIMIT = 5 * 1024 * 1024  # 5 MB — Bedrock limit on base64 string length


def image_to_base64(image_path: Path) -> str:
    """Encode image as base64, compressing if needed to stay under Bedrock's 5 MB limit.

    Bedrock's limit applies to the base64-encoded string length, not the raw byte count.
    A raw image up to ~3.75 MB will base64-encode to just under 5 MB; anything larger
    must be compressed first.
    """
    from PIL import Image
    import io

    raw = image_path.read_bytes()
    encoded = base64.standard_b64encode(raw)
    if len(encoded) <= BEDROCK_IMAGE_LIMIT:
        return encoded.decode()

    # Compress: reduce JPEG quality iteratively until the base64 string fits
    img = Image.open(image_path).convert("RGB")
    for quality in (85, 70, 55, 40):
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=quality, optimize=True)
        data = buf.getvalue()
        encoded = base64.standard_b64encode(data)
        if len(encoded) <= BEDROCK_IMAGE_LIMIT:
            logger.debug("Compressed %s to %.1f MB (quality=%d)",
                         image_path.name, len(data) / 1024 / 1024, quality)
            return encoded.decode()

    # Last resort: scale down by 50%
    img = img.resize((img.width // 2, img.height // 2), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=70)
    data = buf.getvalue()
    logger.warning("Scaled down %s to %.1f MB", image_path.name, len(data) / 1024 / 1024)
    return base64.standard_b64encode(data).decode()


def build_prompt(prior_mapping: list[dict]) -> str:
    if not prior_mapping:
        return BASE_PROMPT.replace("<<PRIOR_MAPPING>>\n", "")
    rows = "\n".join(
        f"  - {r['original_masked']} → {r['replacement']} ({r['type']})"
        for r in prior_mapping
    )
    section = (
        "Previously identified mappings from earlier pages — "
        "use these EXACT replacements for any matching values on this page:\n"
        + rows + "\n\n"
    )
    return BASE_PROMPT.replace("<<PRIOR_MAPPING>>", section.rstrip())


def redact_image(image_path: Path, prior_mapping: list[dict]) -> dict:
    """
    Send a page image to Bedrock.
    Returns dict: {sanitized_text: str, mapping: list[dict]}

    If the model returns non-JSON (e.g. image-only/banner pages),
    the raw response is used as sanitized_text with an empty mapping
    rather than failing the pipeline.
    """
    logger.debug("Encoding image: %s", image_path.name)
    img_b64 = image_to_base64(image_path)
    media_type = "image/jpeg" if img_b64[:4] == "/9j/" else "image/png"
    prompt  = build_prompt(prior_mapping)

    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": MAX_TOKENS,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "image",
                 "source": {"type": "base64", "media_type": media_type, "data": img_b64}},
                {"type": "text", "text": prompt}
            ]
        }]
    }

    last_exc = None
    for attempt in range(1, RETRY_LIMIT + 1):
        try:
            logger.debug("Invoking Bedrock (attempt %d/%d)", attempt, RETRY_LIMIT)
            resp = client.invoke_model(
                modelId=BEDROCK_MODEL, body=json.dumps(body),
                contentType="application/json", accept="application/json"
            )
            raw = json.loads(resp["body"].read())["content"][0]["text"]
            logger.debug("Raw response (first 200 chars): %s", raw[:200].replace("\n", "\\n"))
            return extract_json(raw)

        except json.JSONDecodeError as exc:
            logger.warning(
                "Page %s: non-JSON response (image-only/banner page). "
                "Using raw text, empty mapping. Preview: %s",
                image_path.name, exc.doc[:200].replace("\n", "\\n")
            )
            return {"sanitized_text": exc.doc.strip(), "mapping": []}

        except Exception as exc:
            last_exc = exc
            logger.warning("Attempt %d failed for %s: %s", attempt, image_path.name, exc)
            if attempt < RETRY_LIMIT:
                time.sleep(RETRY_DELAY)

    logger.error("All %d attempts exhausted for %s", RETRY_LIMIT, image_path.name)
    raise last_exc


def merge_mapping(accumulated: list[dict], new_rows: list[dict]) -> list[dict]:
    seen = {(r["original_masked"], r["type"]) for r in accumulated}
    added = 0
    for row in new_rows:
        key = (row["original_masked"], row["type"])
        if key not in seen:
            accumulated.append(row)
            seen.add(key)
            added += 1
    if added:
        logger.debug("Mapping: +%d new entries (total %d)", added, len(accumulated))
    return accumulated


logger.info("Prompt + helper functions defined")

In [ ]:
# Group images by PDF stem
image_files = sorted(TEMP_FOLDER.glob("*.png"))
logger.info("Found %d page image(s) in temp_images/", len(image_files))

grouped: dict[str, list[Path]] = defaultdict(list)
for img in image_files:
    grouped[img.stem.rsplit("_page_", 1)[0]].append(img)

for stem in grouped:
    grouped[stem].sort(key=lambda p: int(p.stem.rsplit("_page_", 1)[-1]))
    logger.info("  %s: %d page(s)", stem, len(grouped[stem]))

In [ ]:
# Run redaction — skips pages that already have a cached .json file
for pdf_stem, page_images in grouped.items():
    logger.info("=== Processing PDF: %s (%d pages) ===", pdf_stem, len(page_images))
    accumulated_mapping: list[dict] = []

    for img_path in page_images:
        cache = TEXT_FOLDER / f"{img_path.stem}.json"

        if cache.exists():
            logger.info("[cache] %s", img_path.name)
            cached = json.loads(cache.read_text(encoding="utf-8"))
            accumulated_mapping = merge_mapping(accumulated_mapping, cached.get("mapping", []))
            continue

        logger.info("Redacting %s ...", img_path.name)
        result = redact_image(img_path, prior_mapping=accumulated_mapping)
        cache.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
        accumulated_mapping = merge_mapping(accumulated_mapping, result.get("mapping", []))
        logger.info("Done: %s — %d mapping entries this page", img_path.name, len(result.get("mapping", [])))

    logger.info("PDF complete: %s — %d unique replacements total", pdf_stem, len(accumulated_mapping))

logger.info("Redaction finished for all PDFs")

In [ ]:
# Preview: first page sanitized text + mapping table for first PDF
json_files = sorted(TEXT_FOLDER.glob("*.json"))
if not json_files:
    logger.warning("No output files found")
else:
    first = json.loads(json_files[0].read_text(encoding="utf-8"))
    logger.info("Preview: %s", json_files[0].name)
    print(f"\n=== {json_files[0].name} — sanitized text ===")
    print(first["sanitized_text"])

    first_stem = json_files[0].stem.rsplit("_page_", 1)[0]
    all_rows: list[dict] = []
    for jf in sorted(TEXT_FOLDER.glob(f"{first_stem}_page_*.json"),
                     key=lambda p: int(p.stem.rsplit("_page_", 1)[-1])):
        data = json.loads(jf.read_text(encoding="utf-8"))
        for row in data.get("mapping", []):
            key = (row["original_masked"], row["type"])
            if key not in {(r["original_masked"], r["type"]) for r in all_rows}:
                all_rows.append(row)

    print(f"\n=== Summary table — {first_stem} ({len(all_rows)} entries) ===")
    print(f"{'Original (masked)':<28} {'Replacement':<28} {'Type'}")
    print("-" * 70)
    for row in all_rows:
        print(f"{row['original_masked']:<28} {row['replacement']:<28} {row['type']}")